In [ ]:
!pip install -q transformers==4.40.0 accelerate datasets
!pip install -q torch scikit-learn pandas numpy matplotlib seaborn openpyxl tqdm psutil datasketch

import os, re, sys, time, json, platform, warnings, zipfile
from pathlib import Path
from collections import Counter

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve,
    average_precision_score, accuracy_score, f1_score
)
from statsmodels.stats.contingency_tables import mcnemar

import psutil
import torch
from torch.utils.data import Dataset, DataLoader
import transformers
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from datasketch import MinHash, MinHashLSH

from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = Path("/content/drive/MyDrive/NLP_Spam/outputs")

In [ ]:
MAIN_SEED   = 42
SEEDS       = [42, 7, 123, 2024, 99]       
LABEL_MAP   = {0: "Legitimate", 1: "Spam"}
N_CLASSES   = 2
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BERT_MODEL  = "distilbert-base-uncased"
MAX_LEN     = 256
BATCH_SIZE  = 16

RUN_FULL_BERT_ABLATION = True

np.random.seed(MAIN_SEED)
torch.manual_seed(MAIN_SEED)

In [ ]:
CSV_PATH = "/content/drive/MyDrive/NLP_Spam/NLP Spam All datasets.csv"

if not Path(CSV_PATH).exists():
    raise FileNotFoundError(f"Dataset not found at: {CSV_PATH}\n")

df_raw = pd.read_csv(CSV_PATH, encoding="utf-8", low_memory=False)

assert "body"  in df_raw.columns, "Missing 'body' column"
assert "label" in df_raw.columns, "Missing 'label' column"

df_raw["label"] = df_raw["label"].astype(int)
df_raw["body"]  = df_raw["body"].fillna("").astype(str)

n_original = len(df_raw)
for lbl, name in LABEL_MAP.items():
    n = (df_raw["label"] == lbl).sum()

In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r"<[^>]+>",             " ",       text)   
    text = re.sub(r"https?://\S+",        " [URL] ", text)   
    text = re.sub(r"\S+@\S+\.\S+",        " [EMAIL] ", text)
    text = re.sub(r"\b\d[\d.,]*\b",       " [NUM] ", text)  
    text = re.sub(r"[^\w\s\[\]]",         " ",       text) 
    text = re.sub(r"\s+",                 " ",       text) 
    return text.strip().lower()


df = df_raw.copy()
df["clean"] = df["body"].apply(clean_text)

n_before_short_filter = len(df)
df = df[df["clean"].str.len() > 10].reset_index(drop=True)
n_removed_short = n_before_short_filter - len(df)

df["subject"]       = df["subject"].fillna("").astype(str)
df["text_combined"] = (df["subject"] + " " + df["clean"]).str.strip()

n_before_exact = len(df)
df = df.drop_duplicates(subset=["text_combined"]).reset_index(drop=True)
n_removed_exact_dupes = n_before_exact - len(df)

def _shingles(text, k=5):
    tokens = text.split()
    if len(tokens) < k:
        return {text}
    return {" ".join(tokens[i:i + k]) for i in range(len(tokens) - k + 1)}

def _minhash(text, num_perm=64):
    m = MinHash(num_perm=num_perm)
    for s in _shingles(text):
        m.update(s.encode("utf8"))
    return m

n_before_near = len(df)
lsh = MinHashLSH(threshold=0.85, num_perm=64)
minhashes = {}
for idx, text in tqdm(df["text_combined"].items(), total=len(df),
                       desc="Building MinHash LSH index"):
    mh = _minhash(text)
    minhashes[idx] = mh

to_drop, seen = set(), set()
for idx in df.index:
    if idx in to_drop:
        continue
    dups = [d for d in lsh.query(minhashes[idx]) if d != idx and d not in to_drop]
    for d in dups:
        to_drop.add(d)
    lsh.insert(idx, minhashes[idx])
    seen.add(idx)

df = df.drop(index=sorted(to_drop)).reset_index(drop=True)
n_removed_near_dupes = n_before_near - len(df)

n_final = len(df)

removal_summary = pd.DataFrame([
    {"stage": "Raw corpus",                       "count": n_original},
    {"stage": "Too-short body removed",            "count": -n_removed_short},
    {"stage": "Exact duplicates removed",          "count": -n_removed_exact_dupes},
    {"stage": "Near-duplicates removed (LSH)",     "count": -n_removed_near_dupes},
    {"stage": "Final retained corpus",             "count": n_final},
])
removal_summary.to_csv(OUTPUT_DIR / "preprocessing_removal_summary.csv", index=False)

In [ ]:
_tok_probe = AutoTokenizer.from_pretrained(BERT_MODEL)
_token_lens = df["text_combined"].sample(
    n=min(2000, len(df)), random_state=MAIN_SEED
).apply(lambda t: len(_tok_probe.encode(t, add_special_tokens=True)))
pct_truncated = (_token_lens > MAX_LEN).mean() * 100

In [ ]:
COLORS = {"Legitimate": "#2ECC71", "Spam": "#E74C3C"}
PLT_STYLE = {
    "figure.facecolor" : "#FAFBFC",
    "axes.facecolor"   : "#FAFBFC",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "DejaVu Sans",
}
plt.rcParams.update(PLT_STYLE)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("SpamAssassin Dataset Overview (post-cleaning, post-dedup)",
             fontsize=13, fontweight="bold", y=1.02)

counts = df["label"].map(LABEL_MAP).value_counts()
bar_colors = [COLORS[c] for c in counts.index]
axes[0].bar(counts.index, counts.values, color=bar_colors, edgecolor="white", width=0.5)
axes[0].set_title("Class Distribution", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Email Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 40, f"{v:,}", ha="center", fontsize=10, fontweight="bold")

src = df.groupby(["source_folder", "label"]).size().unstack(fill_value=0)
src.rename(columns=LABEL_MAP, inplace=True)
src.plot(kind="barh", ax=axes[1], stacked=True,
         color=[COLORS[c] for c in src.columns], edgecolor="white")
axes[1].set_title("Emails per Source Folder", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Count")
axes[1].legend(loc="lower right", fontsize=8)

for lbl, name in LABEL_MAP.items():
    lengths = df[df["label"] == lbl]["body"].str.len().clip(0, 6000)
    axes[2].hist(lengths, bins=50, alpha=0.6, label=name, color=COLORS[name])
axes[2].set_title("Email Body Length", fontsize=11, fontweight="bold")
axes[2].set_xlabel("Characters (clipped at 6,000)")
axes[2].set_ylabel("Count")
axes[2].legend(fontsize=9)

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def stratified_split(dataframe, seed):
    X = dataframe["text_combined"]
    y = dataframe["label"].values
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=seed, stratify=y)
    X_va, X_te, y_va, y_te = train_test_split(
        X_te, y_te, test_size=0.50, random_state=seed, stratify=y_te)
    return X_tr, X_va, X_te, y_tr, y_va, y_te

def report_split_distribution(X_tr, X_va, X_te, y_tr, y_va, y_te, seed):
    for name, X_s, y_s in [("Train", X_tr, y_tr), ("Val", X_va, y_va), ("Test", X_te, y_te)]:
        n = len(X_s)
        n_leg = int((y_s == 0).sum())
        n_spam = int((y_s == 1).sum())

X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(df, MAIN_SEED)
report_split_distribution(X_train, X_val, X_test, y_train, y_val, y_test, MAIN_SEED)

y_bin_test = label_binarize(y_test, classes=[0, 1])
results = {}               
results_multiseed = {}      
cost_log = {}                

In [ ]:
CMAP_BLUE = LinearSegmentedColormap.from_list("bp", ["#EBF5FB", "#1A5276"])
CLS_COLORS = [COLORS["Legitimate"], COLORS["Spam"]]
CLS_NAMES  = ["Legitimate", "Spam"]

def save_fig(fig, name):
    path = OUTPUT_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")

def plot_confusion_matrix(y_true, y_pred, model_name):
    cm   = confusion_matrix(y_true, y_pred, labels=[0, 1])
    norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"Confusion Matrix — {model_name}",
                 fontsize=13, fontweight="bold", y=1.01)

    for ax, data, title, fmt in zip(
            axes, [cm, norm],
            ["Raw Counts", "Normalised (Row %)"], ["d", ".2%"]):
        sns.heatmap(data, annot=True, fmt=fmt, cmap=CMAP_BLUE,
                    xticklabels=CLS_NAMES, yticklabels=CLS_NAMES,
                    linewidths=0.5, linecolor="#CCCCCC", ax=ax,
                    cbar_kws={"shrink": 0.8})
        ax.set_title(title, fontsize=11, pad=8)
        ax.set_xlabel("Predicted", fontsize=10)
        ax.set_ylabel("Actual",    fontsize=10)

    plt.tight_layout()
    save_fig(fig, f"confusion_matrix_{model_name.replace(' ', '_')}.png")
    plt.show()

def plot_roc(y_true, y_proba, model_name):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.set_title(f"ROC / AUC — {model_name}", fontsize=13, fontweight="bold")

    fpr, tpr, _ = roc_curve(y_true, y_proba[:, 1])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=COLORS["Spam"], lw=2.5,
            label=f"Spam  (AUC = {roc_auc:.4f})")
    ax.fill_between(fpr, tpr, alpha=0.10, color=COLORS["Spam"])
    ax.plot([0, 1], [0, 1], "gray", lw=1, linestyle=":")

    ax.set_xlabel("False Positive Rate", fontsize=11)
    ax.set_ylabel("True Positive Rate",  fontsize=11)
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
    plt.tight_layout()
    save_fig(fig, f"roc_{model_name.replace(' ', '_')}.png")
    plt.show()
    return roc_auc

def plot_pr(y_true, y_proba, model_name):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.set_title(f"Precision-Recall — {model_name}", fontsize=13, fontweight="bold")

    prec, rec, _ = precision_recall_curve(y_true, y_proba[:, 1])
    ap           = average_precision_score(y_true, y_proba[:, 1])
    ax.step(rec, prec, color=COLORS["Spam"], lw=2.5, where="post",
            label=f"Spam  (AP = {ap:.4f})")
    ax.fill_between(rec, prec, alpha=0.10, color=COLORS["Spam"], step="post")

    baseline = y_true.mean()
    ax.axhline(baseline, color="gray", lw=1, linestyle=":",
               label=f"Baseline ({baseline:.2f})")

    ax.set_xlabel("Recall",    fontsize=11)
    ax.set_ylabel("Precision", fontsize=11)
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
    plt.tight_layout()
    save_fig(fig, f"pr_{model_name.replace(' ', '_')}.png")
    plt.show()

def plot_training_curves(history, model_name):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
    fig.suptitle(f"Training Curves — {model_name}", fontsize=13, fontweight="bold")

    a1.plot(epochs, history["train_loss"], "o-", color="#3498DB", lw=2, label="Train Loss")
    a1.plot(epochs, history["val_loss"],   "s--", color="#E74C3C", lw=2, label="Val Loss")
    a1.set_title("Loss per Epoch"); a1.set_xlabel("Epoch"); a1.set_ylabel("Cross-Entropy Loss")
    a1.legend(); a1.grid(alpha=0.3); a1.set_xticks(list(epochs))

    a2.plot(epochs, [v * 100 for v in history["val_acc"]], "D-",
            color="#2ECC71", lw=2, label="Val Accuracy")
    a2.set_title("Validation Accuracy"); a2.set_xlabel("Epoch")
    a2.set_ylabel("Accuracy (%)"); a2.set_ylim([80, 101])
    a2.legend(); a2.grid(alpha=0.3); a2.set_xticks(list(epochs))

    plt.tight_layout()
    save_fig(fig, f"training_{model_name.replace(' ', '_')}.png")
    plt.show()

def plot_real_learning_curve(estimator, X_vec, y, model_name):

    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X_vec, y,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=MAIN_SEED),
        train_sizes=np.linspace(0.1, 1.0, 6),
        scoring="f1_weighted", n_jobs=-1, random_state=MAIN_SEED)

    train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
    val_mean, val_std     = val_scores.mean(axis=1), val_scores.std(axis=1)

    fig, ax = plt.subplots(figsize=(8, 5))
    fig.suptitle(f"Learning Curve (5-fold CV) — {model_name}",
                 fontsize=13, fontweight="bold")
    ax.plot(train_sizes, train_mean, "o-", color="#3498DB", lw=2, label="Train F1 (weighted)")
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                     alpha=0.15, color="#3498DB")
    ax.plot(train_sizes, val_mean, "s--", color="#E74C3C", lw=2, label="CV F1 (weighted)")
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                     alpha=0.15, color="#E74C3C")
    ax.set_xlabel("Training examples"); ax.set_ylabel("F1-score (weighted)")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    save_fig(fig, f"learning_curve_{model_name.replace(' ', '_')}.png")
    plt.show()

def plot_feature_importance(vectorizer, classifier, model_name, top_n=25):
    feature_names = np.array(vectorizer.get_feature_names_out())
    clf = classifier
    if hasattr(clf, "named_steps"):
        clf = clf.named_steps.get("clf", clf.named_steps[list(clf.named_steps)[-1]])
    if hasattr(clf, "calibrated_classifiers_"):
        clf = clf.calibrated_classifiers_[0].estimator

    if not hasattr(clf, "coef_"):
        return

    coef = clf.coef_
    if coef.ndim == 2:
        coef = coef[0]

    top_idx  = np.argsort(coef)[-top_n:]
    bot_idx  = np.argsort(coef)[:top_n]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(f"Feature Importance (Top Tokens) — {model_name}",
                 fontsize=13, fontweight="bold")

    axes[0].barh(feature_names[top_idx], coef[top_idx],
                 color=COLORS["Spam"], alpha=0.85, edgecolor="white")
    axes[0].set_title(f"Top {top_n} tokens → Spam", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("TF-IDF Coefficient Weight")

    axes[1].barh(feature_names[bot_idx], coef[bot_idx],
                 color=COLORS["Legitimate"], alpha=0.85, edgecolor="white")
    axes[1].set_title(f"Top {top_n} tokens → Legitimate", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("TF-IDF Coefficient Weight")

    plt.tight_layout()
    save_fig(fig, f"feature_importance_{model_name.replace(' ', '_')}.png")
    plt.show()

def plot_attention_heatmap(tokens, weights, model_name, title_extra=""):
    idx    = np.argsort(weights)[-30:][::-1]
    toks   = [tokens[i] for i in idx]
    wts    = weights[idx]
    wts    = (wts - wts.min()) / (wts.max() - wts.min() + 1e-9)
    cmap   = LinearSegmentedColormap.from_list("attn", ["#EBF5FB", "#2980B9", "#1A5276"])

    fig, ax = plt.subplots(figsize=(15, 3.5))
    ax.set_title(
        f"Attention / Feature Heatmap — {model_name}\n"
        f"Top-30 tokens by weight score  {title_extra}",
        fontsize=12, fontweight="bold")
    im = ax.imshow(wts.reshape(1, -1), cmap=cmap, aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(toks)))
    ax.set_xticklabels(toks, rotation=45, ha="right", fontsize=9)
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, orientation="horizontal", pad=0.35,
                 fraction=0.04, label="Normalised Weight")
    plt.tight_layout()
    save_fig(fig, f"attention_heatmap_{model_name.replace(' ', '_')}.png")
    plt.show()

def plot_cls_report(y_true, y_pred, model_name):
    report = classification_report(
        y_true, y_pred, target_names=CLS_NAMES, output_dict=True)
    rows = []
    for cls in CLS_NAMES + ["macro avg", "weighted avg"]:
        r = report.get(cls, {})
        rows.append([cls,
                     f"{r.get('precision', 0):.4f}",
                     f"{r.get('recall',    0):.4f}",
                     f"{r.get('f1-score',  0):.4f}",
                     int(r.get("support",  0))])

    col_labels = ["Class", "Precision", "Recall", "F1-Score", "Support"]
    row_colors = [["#EAF4FB"] * 5 if i % 2 == 0 else ["#FDFEFE"] * 5
                  for i in range(len(rows))]
    for i in [-2, -1]:
        row_colors[i] = ["#D5EAF7"] * 5

    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.set_title(f"Classification Report — {model_name}",
                 fontsize=12, fontweight="bold", pad=14)
    ax.axis("off")
    tbl = ax.table(cellText=rows, colLabels=col_labels, cellLoc="center",
                   loc="center", cellColours=row_colors)
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 2.0)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor("#CCCCCC")
        if r == 0:
            cell.set_facecolor("#1A5276")
            cell.set_text_props(color="white", fontweight="bold")
    plt.tight_layout()
    save_fig(fig, f"cls_report_{model_name.replace(' ', '_')}.png")
    plt.show()
    pd.DataFrame(
        classification_report(y_true, y_pred, target_names=CLS_NAMES,
                              output_dict=True)
    ).transpose().round(4).to_csv(
        OUTPUT_DIR / f"classification_report_{model_name.replace(' ', '_')}.csv")

def plot_model_comparison(results_dict):
    labels  = list(results_dict.keys())
    metrics = ["accuracy", "f1", "auc"]
    metric_labels = ["Accuracy", "F1-Score", "AUC"]
    x, w = np.arange(len(labels)), 0.25
    bar_colors = ["#3498DB", "#2ECC71", "#E74C3C"]

    def _val(entry, m):
        v = entry[m]
        return v["mean"] if isinstance(v, dict) else v

    def _err(entry, m):
        v = entry[m]
        return v["std"] if isinstance(v, dict) else 0.0

    fig, ax = plt.subplots(figsize=(13, 6))
    ax.set_title("Model Comparison — Accuracy · F1-Score · AUC "
                  "(error bars = ±1 SD across seeds where available)",
                 fontsize=13, fontweight="bold")
    for i, (m, col, ml) in enumerate(zip(metrics, bar_colors, metric_labels)):
        vals = [_val(results_dict[lbl], m) * 100 for lbl in labels]
        errs = [_err(results_dict[lbl], m) * 100 for lbl in labels]
        bars = ax.bar(x + i * w, vals, w, yerr=errs, capsize=3, label=ml, color=col,
                      alpha=0.85, edgecolor="white")
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.3,
                    f"{v:.2f}%", ha="center", va="bottom",
                    fontsize=8, fontweight="bold")

    ax.set_xticks(x + w)
    ax.set_xticklabels(labels, fontsize=9, rotation=15, ha="right")
    ax.set_ylim([60, 105])
    ax.set_ylabel("Score (%)", fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    save_fig(fig, "model_comparison.png")
    plt.show()

def plot_cost_comparison(cost_dict):
    labels = list(cost_dict.keys())
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    fig.suptitle("Computational Cost Comparison", fontsize=13, fontweight="bold")

    train_times = [cost_dict[l]["train_time_s"] for l in labels]
    axes[0].barh(labels, train_times, color="#3498DB", edgecolor="white")
    axes[0].set_title("Training Time (s)"); axes[0].set_xlabel("Seconds")

    latencies = [cost_dict[l]["inference_ms_per_email"] for l in labels]
    axes[1].barh(labels, latencies, color="#2ECC71", edgecolor="white")
    axes[1].set_title("Inference Latency"); axes[1].set_xlabel("ms / email")

    mems = [cost_dict[l]["peak_mem_mb"] for l in labels]
    axes[2].barh(labels, mems, color="#E74C3C", edgecolor="white")
    axes[2].set_title("Peak Memory"); axes[2].set_xlabel("MB")

    for ax in axes:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    plt.tight_layout()
    save_fig(fig, "computational_cost_comparison.png")
    plt.show()
    pd.DataFrame(cost_dict).T.to_csv(OUTPUT_DIR / "computational_cost.csv")


In [ ]:
tfidf = TfidfVectorizer(
    max_features   = 60000,
    ngram_range    = (1, 2),
    sublinear_tf   = True,
    min_df         = 2,
    strip_accents  = "unicode",
    analyzer       = "word",
)
X_train_vec = tfidf.fit_transform(X_train)
X_val_vec   = tfidf.transform(X_val)
X_test_vec  = tfidf.transform(X_test)
feat_names = np.array(tfidf.get_feature_names_out())

In [ ]:
def evaluate_across_seeds(build_and_fit_fn, model_label, seeds=SEEDS):
    accs, f1s, aucs = [], [], []
    for seed in seeds:
        Xtr, Xva, Xte, ytr, yva, yte = stratified_split(df, seed)
        vec = TfidfVectorizer(max_features=60000, ngram_range=(1, 2),
                               sublinear_tf=True, min_df=2,
                               strip_accents="unicode", analyzer="word")
        Xtr_v = vec.fit_transform(Xtr)
        Xte_v = vec.transform(Xte)
        clf = build_and_fit_fn(seed)
        clf.fit(Xtr_v, ytr)
        preds = clf.predict(Xte_v)
        proba = clf.predict_proba(Xte_v)
        accs.append(accuracy_score(yte, preds))
        f1s.append(f1_score(yte, preds, average="weighted"))
        fpr, tpr, _ = roc_curve(yte, proba[:, 1])
        aucs.append(auc(fpr, tpr))

    def _summ(arr):
        arr = np.array(arr)
        mean, std = arr.mean(), arr.std(ddof=1) if len(arr) > 1 else 0.0
        ci95 = 1.96 * std / np.sqrt(len(arr)) if len(arr) > 1 else 0.0
        return {"mean": mean, "std": std, "ci95": ci95, "values": arr.tolist()}

    summary = {"accuracy": _summ(accs), "f1": _summ(f1s), "auc": _summ(aucs)}
    results_multiseed[model_label] = summary

    for m in ["accuracy", "f1", "auc"]:
        s = summary[m]
        print(f"  {m:10s}: {s['mean']*100:.2f}% ± {s['std']*100:.2f}% "
              f"(95% CI: ±{s['ci95']*100:.2f}%)")
    return summary

In [ ]:
t0 = time.time()
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_vec, y_train)
train_time_nb = time.time() - t0

t0 = time.time()
nb_preds = nb.predict(X_test_vec)
nb_proba = nb.predict_proba(X_test_vec)
infer_time_nb = (time.time() - t0) / len(y_test) * 1000

acc = accuracy_score(y_test, nb_preds)
f1  = f1_score(y_test, nb_preds, average="weighted")

plot_confusion_matrix(y_test, nb_preds, "Naive Bayes (NB)")
roc_auc = plot_roc(y_test, nb_proba, "Naive Bayes (NB)")
plot_pr(y_test, nb_proba, "Naive Bayes (NB)")
plot_cls_report(y_test, nb_preds, "Naive Bayes (NB)")

nb_coef = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
top30_idx = np.argsort(nb_coef)[-30:]
plot_attention_heatmap(feat_names[top30_idx].tolist(), nb_coef[top30_idx],
                       "Naive Bayes (NB)",
                       "(log P(spam|token) − log P(legit|token))")

results["Naive Bayes (NB)"] = {"accuracy": acc, "f1": f1, "auc": roc_auc}
cost_log["Naive Bayes (NB)"] = {"train_time_s": train_time_nb,
                                 "inference_ms_per_email": infer_time_nb,
                                 "peak_mem_mb": psutil.Process().memory_info().rss / 1e6}

evaluate_across_seeds(lambda seed: MultinomialNB(alpha=0.1), "Naive Bayes (NB)")

In [ ]:
t0 = time.time()
lr = LogisticRegression(
    C            = 5.0,
    max_iter     = 1000,
    solver       = "saga",
    class_weight = "balanced",
    random_state = MAIN_SEED,
    n_jobs       = -1,
)
lr.fit(X_train_vec, y_train)
train_time_lr = time.time() - t0

t0 = time.time()
lr_preds = lr.predict(X_test_vec)
lr_proba = lr.predict_proba(X_test_vec)
infer_time_lr = (time.time() - t0) / len(y_test) * 1000

acc = accuracy_score(y_test, lr_preds)
f1  = f1_score(y_test, lr_preds, average="weighted")

plot_confusion_matrix(y_test, lr_preds, "Logistic Regression (LR)")
roc_auc = plot_roc(y_test, lr_proba, "Logistic Regression (LR)")
plot_pr(y_test, lr_proba, "Logistic Regression (LR)")

plot_real_learning_curve(
    LogisticRegression(C=5.0, max_iter=1000, solver="saga",
                        class_weight="balanced", random_state=MAIN_SEED),
    X_train_vec, y_train, "Logistic Regression (LR)")

plot_feature_importance(tfidf, lr, "Logistic Regression (LR)")
plot_attention_heatmap(
    feat_names[np.argsort(lr.coef_[0])[-30:]].tolist(),
    np.sort(lr.coef_[0])[-30:],
    "Logistic Regression (LR)",
    "(top spam-class coefficient weights)"
)
plot_cls_report(y_test, lr_preds, "Logistic Regression (LR)")

results["Logistic Regression (LR)"] = {"accuracy": acc, "f1": f1, "auc": roc_auc}
cost_log["Logistic Regression (LR)"] = {"train_time_s": train_time_lr,
                                         "inference_ms_per_email": infer_time_lr,
                                         "peak_mem_mb": psutil.Process().memory_info().rss / 1e6}

evaluate_across_seeds(
    lambda seed: LogisticRegression(C=5.0, max_iter=1000, solver="saga",
                                     class_weight="balanced", random_state=seed, n_jobs=-1),
    "Logistic Regression (LR)")

In [ ]:
t0 = time.time()
svm_base = LinearSVC(C=1.0, max_iter=2000, class_weight="balanced",
                     random_state=MAIN_SEED)
svm = CalibratedClassifierCV(svm_base, cv=3)
svm.fit(X_train_vec, y_train)
train_time_svm = time.time() - t0

t0 = time.time()
svm_preds = svm.predict(X_test_vec)
svm_proba = svm.predict_proba(X_test_vec)
infer_time_svm = (time.time() - t0) / len(y_test) * 1000

acc = accuracy_score(y_test, svm_preds)
f1  = f1_score(y_test, svm_preds, average="weighted")

plot_confusion_matrix(y_test, svm_preds, "Linear SVM")
roc_auc = plot_roc(y_test, svm_proba, "Linear SVM")
plot_pr(y_test, svm_proba, "Linear SVM")
plot_feature_importance(tfidf, svm, "Linear SVM")

coefs = np.mean(
    [cal.estimator.coef_[0] for cal in svm.calibrated_classifiers_],
    axis=0
)
plot_attention_heatmap(
    feat_names[np.argsort(coefs)[-30:]].tolist(),
    np.sort(coefs)[-30:],
    "Linear SVM",
)
plot_cls_report(y_test, svm_preds, "Linear SVM")

results["Linear SVM"] = {"accuracy": acc, "f1": f1, "auc": roc_auc}
cost_log["Linear SVM"] = {"train_time_s": train_time_svm,
                           "inference_ms_per_email": infer_time_svm,
                           "peak_mem_mb": psutil.Process().memory_info().rss / 1e6}

evaluate_across_seeds(
    lambda seed: CalibratedClassifierCV(
        LinearSVC(C=1.0, max_iter=2000, class_weight="balanced", random_state=seed), cv=3),
    "Linear SVM")

In [ ]:
class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length  = self.max_len,
            padding     = "max_length",
            truncation  = True,
            return_tensors = "pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def train_bert_model(model_name, X_tr, y_tr, X_vl, y_vl, X_te,
                     epochs, batch=BATCH_SIZE, lr=2e-5, label="BERT"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSequenceClassification.from_pretrained(
                    model_name, num_labels=N_CLASSES,
                    ignore_mismatched_sizes=True,
                    attn_implementation="eager")
    model.to(DEVICE)
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    train_dl = DataLoader(
        EmailDataset(X_tr, y_tr, tokenizer),
        batch_size=batch, shuffle=True, num_workers=2, pin_memory=True)
    val_dl   = DataLoader(
        EmailDataset(X_vl, y_vl, tokenizer),
        batch_size=batch, shuffle=False, num_workers=2)
    test_dl  = DataLoader(
        EmailDataset(X_te, [0] * len(X_te), tokenizer),
        batch_size=batch, shuffle=False, num_workers=2)

    optimizer    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps  = len(train_dl) * epochs
    scheduler    = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = int(0.1 * total_steps),
        num_training_steps = total_steps)

    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        t_loss = 0.0
        for batch_data in tqdm(train_dl,
                               desc=f"  [{label}] Epoch {epoch+1}/{epochs} train"):
            ids  = batch_data["input_ids"].to(DEVICE)
            mask = batch_data["attention_mask"].to(DEVICE)
            labs = batch_data["label"].to(DEVICE)
            optimizer.zero_grad()
            out  = model(input_ids=ids, attention_mask=mask, labels=labs)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            t_loss += out.loss.item()

        model.eval()
        v_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for batch_data in val_dl:
                ids  = batch_data["input_ids"].to(DEVICE)
                mask = batch_data["attention_mask"].to(DEVICE)
                labs = batch_data["label"].to(DEVICE)
                out  = model(input_ids=ids, attention_mask=mask, labels=labs)
                v_loss  += out.loss.item()
                preds    = out.logits.argmax(dim=1)
                correct += (preds == labs).sum().item()
                total   += labs.size(0)

        atl = t_loss / len(train_dl)
        avl = v_loss / len(val_dl)
        vac = correct / total
        history["train_loss"].append(atl)
        history["val_loss"].append(avl)
        history["val_acc"].append(vac)
    train_time = time.time() - t0

    model.eval()
    all_preds, all_proba = [], []
    t0 = time.time()
    with torch.no_grad():
        for batch_data in tqdm(test_dl, desc=f"  [{label}] Inference"):
            ids  = batch_data["input_ids"].to(DEVICE)
            mask = batch_data["attention_mask"].to(DEVICE)
            out  = model(input_ids=ids, attention_mask=mask)
            probs = torch.softmax(out.logits, dim=1).cpu().numpy()
            all_preds.extend(probs.argmax(axis=1).tolist())
            all_proba.extend(probs.tolist())
    infer_time_ms_per_email = (time.time() - t0) / len(X_te) * 1000

    if DEVICE == "cuda":
        peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6
    else:
        peak_mem_mb = psutil.Process().memory_info().rss / 1e6

    spam_idx  = next((i for i, l in enumerate(y_te_global) if l == 1), 0)
    sample    = X_te_global.iloc[spam_idx]
    enc       = tokenizer(sample, max_length=64, truncation=True,
                          return_tensors="pt", padding="max_length")
    with torch.no_grad():
        attn_out = model(input_ids=enc["input_ids"].to(DEVICE),
                         attention_mask=enc["attention_mask"].to(DEVICE),
                         output_attentions=True)
    attn_mat    = attn_out.attentions[-1][0].mean(dim=0)[0].cpu().numpy()
    raw_tokens  = tokenizer.convert_ids_to_tokens(enc["input_ids"][0].numpy())
    valid       = [(t, float(w)) for t, w in zip(raw_tokens, attn_mat)
                   if t not in ("[PAD]", "<pad>", "[SEP]", "[CLS]")]
    attn_tokens  = [t for t, _ in valid]
    attn_weights = np.array([w for _, w in valid])

    cost = {"train_time_s": train_time,
            "inference_ms_per_email": infer_time_ms_per_email,
            "peak_mem_mb": peak_mem_mb}

    return (np.array(all_preds), np.array(all_proba),
            history, attn_tokens, attn_weights, cost)


X_te_global = X_test
y_te_global = y_test

In [ ]:
if RUN_FULL_BERT_ABLATION:
    bert_configs = [
        {"epochs": 2,  "lr": 2e-5, "label": "DistilBERT (2 epochs, lr=2e-5)"},
        {"epochs": 5,  "lr": 2e-5, "label": "DistilBERT (5 epochs, lr=2e-5)"},
        {"epochs": 10, "lr": 2e-5, "label": "DistilBERT (10 epochs, lr=2e-5)"},
    ]
else:
    bert_configs = [
        {"epochs": 2, "lr": 3e-5, "label": "DistilBERT (2 epochs, lr=3e-5)"},
        {"epochs": 3, "lr": 2e-5, "label": "DistilBERT (3 epochs, lr=2e-5)"},
    ]

bert_run_details = {}

for cfg in bert_configs:
    label = cfg["label"]
    (preds, proba, hist, attn_tokens, attn_weights, cost) = train_bert_model(
        BERT_MODEL, X_train, y_train, X_val, y_val, X_test,
        epochs=cfg["epochs"], batch=BATCH_SIZE, lr=cfg["lr"], label=label
    )

    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average="weighted")
    fpr, tpr, _ = roc_curve(y_test, proba[:, 1])
    roc_auc = auc(fpr, tpr)

    plot_confusion_matrix(y_test, preds, label)
    plot_roc(y_test, proba, label)
    plot_pr(y_test, proba, label)
    plot_training_curves(hist, label)
    plot_attention_heatmap(attn_tokens, attn_weights, label,
                           "(CLS token attention — first spam sample)")
    plot_cls_report(y_test, preds, label)

    results[label] = {"accuracy": acc, "f1": f1, "auc": roc_auc}
    cost_log[label] = cost
    bert_run_details[label] = {"preds": preds, "proba": proba}


In [ ]:
best_bert_label = max(
    (lbl for lbl in bert_run_details), key=lambda lbl: results[lbl]["accuracy"])
bert_preds_best = bert_run_details[best_bert_label]["preds"]

svm_correct  = (svm_preds == y_test)
bert_correct = (bert_preds_best == y_test)

both_correct    = int(np.sum(svm_correct & bert_correct))
svm_only_right  = int(np.sum(svm_correct & ~bert_correct))
bert_only_right = int(np.sum(~svm_correct & bert_correct))
both_wrong      = int(np.sum(~svm_correct & ~bert_correct))

contingency = [[both_correct, svm_only_right],
               [bert_only_right, both_wrong]]

mcnemar_result = mcnemar(contingency, exact=(svm_only_right + bert_only_right) < 25,
                          correction=True)


with open(OUTPUT_DIR / "mcnemar_test.json", "w") as f:
    json.dump({
        "compared": ["Linear SVM", best_bert_label],
        "contingency_table": contingency,
        "statistic": float(mcnemar_result.statistic),
        "p_value": float(mcnemar_result.pvalue),
    }, f, indent=2)

In [ ]:
plot_model_comparison(results)
plot_cost_comparison(cost_log)

df_res = pd.DataFrame(results).T
df_res.columns = ["Accuracy", "F1-score (weighted)", "AUC"]
df_res.to_csv(OUTPUT_DIR / "summary_results_main_seed.csv")

rows = []
for model, summ in results_multiseed.items():
    rows.append({
        "model": model,
        "accuracy_mean": summ["accuracy"]["mean"], "accuracy_std": summ["accuracy"]["std"],
        "accuracy_ci95": summ["accuracy"]["ci95"],
        "f1_mean": summ["f1"]["mean"], "f1_std": summ["f1"]["std"], "f1_ci95": summ["f1"]["ci95"],
        "auc_mean": summ["auc"]["mean"], "auc_std": summ["auc"]["std"], "auc_ci95": summ["auc"]["ci95"],
    })
df_multiseed = pd.DataFrame(rows)
df_multiseed.to_csv(OUTPUT_DIR / "summary_results_multiseed.csv", index=False)

for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size / 1024